In [ ]:
import os
import pandas as pd
import json

pd.set_option('display.max_columns', None)

# Common Functions

In [4]:
def get_infractions(record):
    infraction_counts = {
        'collisions_layout': 0.0,
        'collisions_pedestrian': 0.0,
        'collisions_vehicle': 0.0,
        'outside_route_lanes': 0.0,
        'red_light': 0.0,
        'route_dev': 0.0,
        'route_timeout': 0.0,
        'stop_infraction': 0.0,
        'vehicle_blocked': 0.0
    }
    for key, value in record["infractions"].items():
        infraction_counts[key] += len(value)
    return infraction_counts

In [ ]:
def get_route_df(exp_path):
    # Define the column names
    columns = [
        'approach', 'run', 'route', 'score_composed', 'score_penalty', 'score_route',
        'collisions_layout', 'collisions_pedestrian', 'collisions_vehicle', 'outside_route_lanes',
        'red_light', 'route_dev', 'route_timeout', 'stop_infraction', 'vehicle_blocked'
    ]

    # Create an empty DataFrame with the specified columns
    df = pd.DataFrame(columns=columns)

    for approach in os.listdir(exp_path):
        for run in range(1, 6):
            results_path = f"{exp_path}/{approach}/results_run{run}.json"
            # Read json file
            with open(results_path) as f:
                results = json.load(f)
            if len(results['_checkpoint']['records']) < 10:
                print(f"Only {len(results['_checkpoint']['records'])} results in {results_path}")
            for idx in range(len(results['_checkpoint']['records'])):
                record = results['_checkpoint']['records'][idx]
                infraction_counts = get_infractions(record)
                scores = record["scores"]
                
                row = [approach, run, idx]
                row.extend(list(scores.values()))
                row.extend(list(infraction_counts.values()))
                df.loc[len(df)] = row
    return df

In [ ]:
def format_summary_for_latex(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert a summary df with MultiIndex columns (metric, mean/std)
    into a latex-ready DataFrame with formatted mean ± std,
    replace column names with LaTeX labels, and drop any column
    not in the allowed list.
    """
    # Mapping of raw metric names → LaTeX labels
    col_names = {
        'score_composed': r'\makecell{Driving \\ Score} $\uparrow$',
        'collisions_pedestrian': r'\makecell{Collision \\ Pedestrians} $\downarrow$',
        'collisions_vehicle': r'\makecell{Collision \\ Vehicles} $\downarrow$',
        'red_light': r'\makecell{Red Light \\ Infraction} $\downarrow$',
        'stop_infraction': r'\makecell{Stop Sign \\ Infraction} $\downarrow$',
        'route_timeout': r'\makecell{Route \\ Timeout} $\downarrow$',
        'vehicle_blocked': r'\makecell{Vehicle \\ Blocked} $\downarrow$',
    }

    allowed_metrics = set(col_names.keys())
    formatted = pd.DataFrame(index=df.index)

    if isinstance(df.columns, pd.MultiIndex):
        for metric in df.columns.get_level_values(0).unique():
            if metric not in allowed_metrics:
                continue  # drop columns not in allowed list
            mean_vals = df[(metric, "mean")]
            std_vals = df[(metric, "std")]
            label = col_names[metric]
            formatted[label] = [
                f"{m:.2f}$\\pm${s:.2f}" for m, s in zip(mean_vals, std_vals)
            ]
    else:
        # Handle flat structure with mean/std as rows
        if "mean" in df.index and "std" in df.index:
            df_t = df.T
            df_multi = df_t.stack().to_frame().T
            df_multi.index = ["Baseline"]
            return format_summary_for_latex(df_multi)
        raise ValueError("Unsupported dataframe structure")

    # Insert index as first column
    formatted.insert(0, "Treatment", formatted.index)
    formatted["Treatment"] = formatted["Treatment"].replace({
        "gt": "M4PC (GT)",
        "sgg": "M4PC (SGG)",
        "t4pc": "T4PC",
        "original": "Baseline",
    })
    return formatted.reset_index(drop=True)

# TCP

In [27]:
tcp_raw_df = get_route_df("exp_results/tcp")

In [28]:
tcp_summary = tcp_raw_df.drop(columns=["run","route"]).groupby(['approach']).agg(['mean','std'])
tcp_summary

score_composed            score_penalty           score_route  \
                   mean        std          mean       std        mean   
approach                                                                 
gt            87.156257  24.759307      0.937800  0.135002   93.376257   
original      76.860800  23.434110      0.768608  0.234341  100.000000   
sgg           79.989379  27.966469      0.865285  0.214443   93.460910   
t4pc          78.231143  21.756072      0.789952  0.203597   98.726572   

                    collisions_layout           collisions_pedestrian       \
                std              mean       std                  mean  std   
approach                                                                     
gt        22.690252              0.02  0.141421                   0.0  0.0   
original   0.000000              0.00  0.000000                   0.0  0.0   
sgg       22.406386              0.02  0.141421                   0.0  0.0   
t4pc       9.004492              0.00  0.000000                   0.0  0.0   

         collisions_vehicle           outside_route_lanes           red_light  \
                       mean       std                mean       std      mean   
approach                                                                        
gt                     0.04  0.197949                0.00  0.000000      0.06   
original               0.10  0.364216                0.00  0.000000      0.06   
sgg                    0.22  0.545482                0.06  0.239898      0.06   
t4pc                   0.12  0.328261                0.00  0.000000      0.00   

                   route_dev      route_timeout      stop_infraction  \
               std      mean  std          mean  std            mean   
approach                                                               
gt        0.239898       0.0  0.0           0.0  0.0            0.12   
original  0.239898       0.0  0.0           0.0  0.0            1.10   
sgg       0.313636       0.0  0.0           0.0  0.0            0.20   
t4pc      0.000000       0.0  0.0           0.0  0.0            0.94   

                   vehicle_blocked            
               std            mean       std  
approach                                      
gt        0.328261            0.08  0.274048  
original  1.233048            0.00  0.000000  
sgg       0.404061            0.08  0.274048  
t4pc      1.132272            0.02  0.141421

In [29]:
formated_table = format_summary_for_latex(tcp_summary)
print(formated_table.to_latex(index=False, escape=False))

\begin{tabular}{llllllll}
\toprule
Treatment & \makecell{Driving \\ Score} $\uparrow$ & \makecell{Collision \\ Pedestrians} $\downarrow$ & \makecell{Collision \\ Vehicles} $\downarrow$ & \makecell{Red Light \\ Infraction} $\downarrow$ & \makecell{Route \\ Timeout} $\downarrow$ & \makecell{Stop Sign \\ Infraction} $\downarrow$ & \makecell{Vehicle \\ Blocked} $\downarrow$ \\
\midrule
M4PC (GT) & 87.16$\pm$24.76 & 0.00$\pm$0.00 & 0.04$\pm$0.20 & 0.06$\pm$0.24 & 0.00$\pm$0.00 & 0.12$\pm$0.33 & 0.08$\pm$0.27 \\
Baseline & 76.86$\pm$23.43 & 0.00$\pm$0.00 & 0.10$\pm$0.36 & 0.06$\pm$0.24 & 0.00$\pm$0.00 & 1.10$\pm$1.23 & 0.00$\pm$0.00 \\
M4PC (SGG) & 79.99$\pm$27.97 & 0.00$\pm$0.00 & 0.22$\pm$0.55 & 0.06$\pm$0.31 & 0.00$\pm$0.00 & 0.20$\pm$0.40 & 0.08$\pm$0.27 \\
T4PC & 78.23$\pm$21.76 & 0.00$\pm$0.00 & 0.12$\pm$0.33 & 0.00$\pm$0.00 & 0.00$\pm$0.00 & 0.94$\pm$1.13 & 0.02$\pm$0.14 \\
\bottomrule
\end{tabular}



# INTERFUSER

In [30]:
interfuser_raw_df = get_route_df("exp_results/interfuser")

In [34]:
interfuser_summary = interfuser_raw_df.drop(columns=["run","route"]).groupby(['approach']).agg(['mean','std'])
interfuser_summary

score_composed            score_penalty           score_route  \
                   mean        std          mean       std        mean   
approach                                                                 
gt            69.741298  32.421579      0.832840  0.253823   85.068720   
original      59.407748  32.277807      0.610170  0.311025   96.697223   
sgg           60.137841  32.366761      0.676468  0.302368   91.243849   
t4pc          64.050272  35.473673      0.681382  0.346092   93.833758   

                    collisions_layout           collisions_pedestrian  \
                std              mean       std                  mean   
approach                                                                
gt        27.714856              0.00  0.000000                  0.10   
original  13.990829              0.00  0.000000                  0.26   
sgg       25.623821              0.02  0.141421                  0.12   
t4pc      17.296467              0.00  0.000000                  0.18   

                   collisions_vehicle           outside_route_lanes       \
               std               mean       std                mean  std   
approach                                                                   
gt        0.364216               0.34  0.592814                 0.0  0.0   
original  0.486973               0.64  0.875051                 0.0  0.0   
sgg       0.385450               0.54  0.787919                 0.0  0.0   
t4pc      0.388088               0.72  1.069808                 0.0  0.0   

         red_light           route_dev      route_timeout            \
              mean       std      mean  std          mean       std   
approach                                                              
gt            0.04  0.197949       0.0  0.0          0.02  0.141421   
original      0.42  0.672795       0.0  0.0          0.06  0.239898   
sgg           0.42  0.641745       0.0  0.0          0.04  0.197949   
t4pc          0.28  0.496518       0.0  0.0          0.18  0.388088   

         stop_infraction      vehicle_blocked            
                    mean  std            mean       std  
approach                                                 
gt                   0.0  0.0            0.26  0.443087  
original             0.0  0.0            0.00  0.000000  
sgg                  0.0  0.0            0.12  0.328261  
t4pc                 0.0  0.0            0.00  0.000000

In [35]:
formated_table = format_summary_for_latex(interfuser_summary)
print(formated_table.to_latex(index=False, escape=False))

\begin{tabular}{llllllll}
\toprule
Treatment & \makecell{Driving \\ Score} $\uparrow$ & \makecell{Collision \\ Pedestrians} $\downarrow$ & \makecell{Collision \\ Vehicles} $\downarrow$ & \makecell{Red Light \\ Infraction} $\downarrow$ & \makecell{Route \\ Timeout} $\downarrow$ & \makecell{Stop Sign \\ Infraction} $\downarrow$ & \makecell{Vehicle \\ Blocked} $\downarrow$ \\
\midrule
M4PC (GT) & 69.74$\pm$32.42 & 0.10$\pm$0.36 & 0.34$\pm$0.59 & 0.04$\pm$0.20 & 0.02$\pm$0.14 & 0.00$\pm$0.00 & 0.26$\pm$0.44 \\
Baseline & 59.41$\pm$32.28 & 0.26$\pm$0.49 & 0.64$\pm$0.88 & 0.42$\pm$0.67 & 0.06$\pm$0.24 & 0.00$\pm$0.00 & 0.00$\pm$0.00 \\
M4PC (SGG) & 60.14$\pm$32.37 & 0.12$\pm$0.39 & 0.54$\pm$0.79 & 0.42$\pm$0.64 & 0.04$\pm$0.20 & 0.00$\pm$0.00 & 0.12$\pm$0.33 \\
T4PC & 64.05$\pm$35.47 & 0.18$\pm$0.39 & 0.72$\pm$1.07 & 0.28$\pm$0.50 & 0.18$\pm$0.39 & 0.00$\pm$0.00 & 0.00$\pm$0.00 \\
\bottomrule
\end{tabular}



# PYLOT

In [36]:
pylot_raw_df = get_route_df("exp_results/pylot")

In [37]:
pylot_summary = pylot_raw_df.drop(columns=["run","route"]).groupby(['approach']).agg(['mean', 'std'])
pylot_summary

score_composed            score_penalty           score_route  \
                   mean        std          mean       std        mean   
approach                                                                 
gt            80.525713  24.025904      0.826946  0.224341   96.962105   
original      68.956993  26.697124      0.695268  0.263332   98.954387   
sgg           75.375870  27.045437      0.757358  0.266229   99.089405   

                   collisions_layout      collisions_pedestrian       \
               std              mean  std                  mean  std   
approach                                                               
gt        8.349455               0.0  0.0                   0.0  0.0   
original  4.348000               0.0  0.0                   0.0  0.0   
sgg       2.352369               0.0  0.0                   0.0  0.0   

         collisions_vehicle           outside_route_lanes           red_light  \
                       mean       std                mean       std      mean   
approach                                                                        
gt                     0.20  0.451754                0.02  0.141421      0.38   
original               0.32  0.652781                0.00  0.000000      0.54   
sgg                    0.24  0.476381                0.00  0.000000      0.52   

                   route_dev           route_timeout            \
               std      mean       std          mean       std   
approach                                                         
gt        0.635353      0.10  0.303046          0.12  0.328261   
original  0.862128      0.08  0.274048          0.00  0.000000   
sgg       0.994680      0.10  0.303046          0.04  0.197949   

         stop_infraction           vehicle_blocked            
                    mean       std            mean       std  
approach                                                      
gt                  0.00  0.000000            0.00  0.000000  
original            0.44  0.540597            0.02  0.141421  
sgg                 0.24  0.431419            0.00  0.000000

In [38]:
formated_table = format_summary_for_latex(pylot_summary)
print(formated_table.to_latex(index=False, escape=False))

\begin{tabular}{llllllll}
\toprule
Treatment & \makecell{Driving \\ Score} $\uparrow$ & \makecell{Collision \\ Pedestrians} $\downarrow$ & \makecell{Collision \\ Vehicles} $\downarrow$ & \makecell{Red Light \\ Infraction} $\downarrow$ & \makecell{Route \\ Timeout} $\downarrow$ & \makecell{Stop Sign \\ Infraction} $\downarrow$ & \makecell{Vehicle \\ Blocked} $\downarrow$ \\
\midrule
M4PC (GT) & 80.53$\pm$24.03 & 0.00$\pm$0.00 & 0.20$\pm$0.45 & 0.38$\pm$0.64 & 0.12$\pm$0.33 & 0.00$\pm$0.00 & 0.00$\pm$0.00 \\
Baseline & 68.96$\pm$26.70 & 0.00$\pm$0.00 & 0.32$\pm$0.65 & 0.54$\pm$0.86 & 0.00$\pm$0.00 & 0.44$\pm$0.54 & 0.02$\pm$0.14 \\
M4PC (SGG) & 75.38$\pm$27.05 & 0.00$\pm$0.00 & 0.24$\pm$0.48 & 0.52$\pm$0.99 & 0.04$\pm$0.20 & 0.24$\pm$0.43 & 0.00$\pm$0.00 \\
\bottomrule
\end{tabular}

